# Epi Info AI Cohort or Cross-Sectional validation lab - V0.13

Validate the candidate `epi.sampleSize.cohortCrossSectional` Rust/WebAssembly kernel against an independent translation of CDC's legacy formulas. Passing is evidence, not statistical approval; G5 remains consolidated.

In [ ]:
import math
from pyodide.http import pyfetch
from js import WebAssembly, Uint8Array
from scipy.stats import norm
fixture_response = await pyfetch('/validation-fixtures/cohort-cross-sectional-v0.13.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
wasm_response = await pyfetch('/epi2x2.wasm')
instance = await WebAssembly.instantiate(Uint8Array.new(await wasm_response.buffer()), {})
rust = instance.instance.exports


In [ ]:
def legacy_norm_tail(z):
    z = abs(z); p = 1 + z*(0.04986735 + z*(0.02114101 + z*(0.00327763 + z*(0.0000380036 + z*(0.0000488906 + z*0.000005383)))))
    p=p*p; p=p*p; p=p*p; return 1/(p*p)
def legacy_anorm(p):
    v=dv=.5; z=0
    while dv > 1e-6:
        z=1/v-1; dv/=2; v=v-dv if legacy_norm_tail(z)>p else v+dv
    return z
def legacy_methods(i):
    ratio=i['unexposedToExposedRatio']; u=i['unexposedOutcomePercent']/100; odds=i['oddsRatio']; e=u*odds/(1+u*(odds-1))
    za=legacy_anorm(1-i['confidenceLevel']); power=i['powerPercent']/100
    zb=-legacy_anorm(2*power) if power < .5 else legacy_anorm(2-2*power)
    pbar=(e+ratio*u)/(1+ratio); qbar=1-pbar; difference=e-u
    k=(za+zb)**2*pbar*qbar*(ratio+1)/(difference**2*ratio)
    f=(za*math.sqrt((ratio+1)*pbar*qbar)+zb*math.sqrt(ratio*e*(1-e)+u*(1-u)))**2/(ratio*difference**2)
    cc=f*(1+math.sqrt(1+2*(ratio+1)/(f*ratio*abs(difference))))**2/4
    labels=['Kelsey','Fleiss','Fleiss with continuity correction']
    return [{'method': label, 'exposed': math.ceil(raw), 'unexposed': math.ceil(raw*ratio), 'total': math.ceil(raw)+math.ceil(raw*ratio)} for label,raw in zip(labels,[k,f,cc])]
for case in fixture['cases']: assert legacy_methods(case['input']) == case['methods']
print('PASS: independent Python translation matches both frozen legacy-formula cases')


In [ ]:
case=fixture['cases'][0]; level=case['input']['confidenceLevel']; power=case['input']['powerPercent']/100
{'legacy_confidence_z': legacy_anorm(1-level), 'scipy_confidence_z': norm.ppf((1+level)/2), 'legacy_power_z': legacy_anorm(2-2*power), 'scipy_power_z': norm.ppf(power)}


In [ ]:
def wasm_methods(i):
    result=[]; labels=['Kelsey','Fleiss','Fleiss with continuity correction']
    for method,label in enumerate(labels):
        args=(i['confidenceLevel'],i['powerPercent'],i['unexposedToExposedRatio'],i['unexposedOutcomePercent']/100,i['oddsRatio'])
        exposed=int(rust.cohort_sample_size(method,0,*args)); unexposed=int(rust.cohort_sample_size(method,1,*args))
        result.append({'method':label,'exposed':exposed,'unexposed':unexposed,'total':exposed+unexposed})
    return result
for case in fixture['cases']: assert wasm_methods(case['input']) == case['methods']
assert math.isnan(float(rust.cohort_sample_size(0,0,.95,80,1,.05,1)))
print('PASS: deployed Rust/WASM matches equal and unequal groups and rejects a no-effect design')
